<a href="https://colab.research.google.com/github/nirmalpate/RAG-BASED-CHATBOT/blob/main/RAG_BASED_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## RAG BASED MODEL.

In [ ]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb scikit-learn numpy

In [ ]:
import os
import uuid
from typing import List, Dict, Any, Tuple
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyMuPDFLoader
from sentence_transformers import SentenceTransformer
import chromadb
import os
import uuid
from typing import List, Dict, Any
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

In [ ]:
# 1. Initialize local persistent ChromaDB storage
# Documents and embeddings will be stored on disk in the './chroma_db' folder
chroma_client = chromadb.PersistentClient(path="./chroma_db")

In [ ]:
# 2. Get or create a collection (equivalent to a table in relational databases)
collection = chroma_client.get_or_create_collection(
    name="rag_knowledge_base",
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity for vector comparison
)

In [ ]:
# 3. Load Sentence Transformer model for generating text embeddings
# 'all-MiniLM-L6-v2' maps sentences/documents to a 384-dimensional dense vector space
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyMuPDFLoader
from langchain_core.documents import Document

In [ ]:
def load_documents_from_directory(dir_path: str) -> List[Document]:
  if not os.path.exists(dir_path):
        raise FileNotFoundError(f"Directory '{dir_path}' does not exist.")
  documents: List[Document] = []
  txt_loader = DirectoryLoader(
      dir_path,
      glob="**/*.txt",
      loader_cls=TextLoader,
      loader_kwargs={"encoding": "utf-8"}
  )
  txt_docs = txt_loader.load()
  documents.extend(txt_docs)

  print(f"Loaded {len(documents)} raw document page(s) from '{dir_path}'.")
  return documents

In [ ]:
##Step 3: Split Documents into Chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_documents(documents: List[Document], chunk_size: int = 500, chunk_overlap: int = 50) -> List[Document]:
  text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,       # Maximum characters per chunk
        chunk_overlap=chunk_overlap, # Overlap between adjacent chunks to maintain context
        separators=["\n\n", "\n", " ", ""] # Order of split points
    )
  chunks = text_splitter.split_documents(documents)
  print(f"Split {len(documents)} documents into {len(chunks)} text chunks.")
  return chunks

In [ ]:
def ingest_chunks_to_chroma(chunks: List[Document], batch_size: int = 64) -> None:
  if not chunks:
        print("No chunks to ingest.")
        return
  total_chunks = len(chunks)

  # Process in batches for better memory efficiency and performance
  for i in range(0, total_chunks, batch_size):
    batch = chunks[i : i + batch_size]
    texts = [doc.page_content for doc in batch] # Extract text content from documents
    embeddings = embedding_model.encode(texts, show_progress_bar=False).tolist()
    ids = [str(uuid.uuid4()) for _ in batch]
    # Prepare metadata dictionaries (convert non-primitive types if any)
    metadatas = [doc.metadata for doc in batch]
    # Upsert into ChromaDB collection
    collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )
    print(f"Ingested batch {i // batch_size + 1}/{(total_chunks + batch_size - 1) // batch_size} ({len(batch)} chunks)")

  print(f"\nSuccessfully ingested all {total_chunks} chunks into ChromaDB!")

In [ ]:
if __name__ == "__main__":
    # Define document folder path
    data_directory = "./sample_data"

    # Create directory if it doesn't exist for demo purposes
    if not os.path.exists(data_directory):
        os.makedirs(data_directory)
        with open(f"{data_directory}/sample.txt", "w", encoding="utf-8") as f:
            f.write("Retrieval-Augmented Generation (RAG) combines search algorithms with LLMs.\n" * 20)

    # Execute Pipeline
    print("--- Starting Data Ingestion Pipeline ---")

    # 1. Load
    raw_docs = load_documents_from_directory(data_directory)

    # 2. Chunk
    doc_chunks = split_documents(raw_docs, chunk_size=300, chunk_overlap=30)

    # 3. Embed & Store
    ingest_chunks_to_chroma(doc_chunks)

    # Verify collection count
    print(f"Total stored vectors in collection: {collection.count()}")

In [ ]:
def retrieve_top_k_chunks(query_text: str, top_k: int = 3) -> List[Dict[str, Any]]:
    query_embedding = embedding_model.encode(query_text).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_chunks = []

    # Safely extract first element from nested lists
    if results and results.get("documents") and len(results["documents"]) > 0:
        documents = results["documents"][0]
        metadatas = results["metadatas"][0] if results.get("metadatas") else [{}] * len(documents)
        distances = results["distances"][0] if results.get("distances") else [0.0] * len(documents)

        for doc, meta, dist in zip(documents, metadatas, distances):
            retrieved_chunks.append({
                "chunk_text": doc,
                "metadata": meta,
                "chroma_distance": dist
            })

    return retrieved_chunks

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def rank_chunks_with_cosine_similarity(
    query_text: str,
    candidate_chunks: List[str]
) -> List[Tuple[str, float]]:
    """
    Computes exact pairwise cosine similarity scores between a query and a list of text chunks.
    """
    if not candidate_chunks:
        return []

    # 1. Embed query text and reshape to a 2D matrix (1, vector_dim)
    query_vec = embedding_model.encode([query_text])

    # 2. Embed candidate text chunks into a 2D matrix (n_chunks, vector_dim)
    chunk_vecs = embedding_model.encode(candidate_chunks)

    # 3. Compute pairwise cosine similarity matrix
    # Output shape: (1, n_chunks)
    similarity_matrix = cosine_similarity(query_vec, chunk_vecs)

    # 4. Extract 1D array of scores
    similarity_scores = similarity_matrix[0]

    # 5. Pair text chunks with their similarity scores and sort in descending order
    ranked_results = list(zip(candidate_chunks, similarity_scores))
    ranked_results.sort(key=lambda x: x[1], reverse=True)

    return ranked_results

In [50]:
def retrieve_top_k_chunks(query_text: str, top_k: int = 3) -> List[Dict[str, Any]]:
    query_embedding = embedding_model.encode(query_text).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_chunks = []

    # Safely extract first element from nested lists
    if results and results.get("documents") and len(results["documents"]) > 0:
        documents = results["documents"][0]
        metadatas = results["metadatas"][0] if results.get("metadatas") else [{}] * len(documents)
        distances = results["distances"][0] if results.get("distances") else [0.0] * len(documents)

        for doc, meta, dist in zip(documents, metadatas, distances):
            retrieved_chunks.append({
                "chunk_text": doc,
                "metadata": meta,
                "chroma_distance": dist
            })

    return retrieved_chunks

In [49]:
def retrieve_top_k_chunks(query_text: str, top_k: int = 3) -> List[Dict[str, Any]]:

    # --------------------------------------------------------------------------
    # 1. SENTENCE TRANSFORMER CONVERTS QUERY TO EMBEDDING VECTOR
    # --------------------------------------------------------------------------
    # The .encode() method takes the raw text string (e.g., "What is RAG?")
    # and generates a 384-dimensional dense float vector array.
    query_embedding = embedding_model.encode(query_text).tolist()


    # --------------------------------------------------------------------------
    # 2. CHROMADB SEARCHES & RETURNS THE MATCHING CHUNKS (REPLY)
    # --------------------------------------------------------------------------
    # Passes the generated query vector into ChromaDB's .query() method.
    # ChromaDB traverses its HNSW graph and returns the top-K closest document chunks.
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    return results

In [51]:
import os
import uuid
from typing import List, Dict, Any, Tuple
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize local persistent ChromaDB storage
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# 2. Get or create a collection (uses cosine similarity)
collection = chroma_client.get_or_create_collection(
    name="rag_knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

# 3. Load Sentence Transformer model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


def load_documents_from_directory(dir_path: str) -> List[Document]:
    if not os.path.exists(dir_path):
        raise FileNotFoundError(f"Directory '{dir_path}' does not exist.")

    txt_loader = DirectoryLoader(
        dir_path,
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    documents = txt_loader.load()
    print(f"Loaded {len(documents)} raw document page(s) from '{dir_path}'.")
    return documents


def split_documents(
    documents: List[Document],
    chunk_size: int = 300,
    chunk_overlap: int = 30
) -> List[Document]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(chunks)} text chunks.")
    return chunks


def ingest_chunks_to_chroma(chunks: List[Document], batch_size: int = 64) -> None:
    if not chunks:
        print("No chunks to ingest.")
        return

    total_chunks = len(chunks)
    for i in range(0, total_chunks, batch_size):
        batch = chunks[i : i + batch_size]
        texts = [doc.page_content for doc in batch]
        embeddings = embedding_model.encode(texts, show_progress_bar=False).tolist()
        ids = [str(uuid.uuid4()) for _ in batch]
        metadatas = [doc.metadata for doc in batch]

        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )
        print(f"Ingested batch {i // batch_size + 1}/{(total_chunks + batch_size - 1) // batch_size} ({len(batch)} chunks)")

    print(f"\nSuccessfully ingested all {total_chunks} chunks into ChromaDB!")


def retrieve_top_k_chunks(query_text: str, top_k: int = 3) -> List[Dict[str, Any]]:
    query_embedding = embedding_model.encode(query_text).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_chunks = []
    if results and results.get("documents") and results["documents"][0]:
        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]

        for doc, meta, dist in zip(documents, metadatas, distances):
            retrieved_chunks.append({
                "chunk_text": doc,
                "metadata": meta,
                "chroma_distance": dist
            })

    return retrieved_chunks


def rank_chunks_with_cosine_similarity(
    query_text: str,
    candidate_chunks: List[str]
) -> List[Tuple[str, float]]:
    if not candidate_chunks:
        return []

    query_vec = embedding_model.encode([query_text])
    chunk_vecs = embedding_model.encode(candidate_chunks)
    similarity_matrix = cosine_similarity(query_vec, chunk_vecs)
    similarity_scores = similarity_matrix[0]

    ranked_results = list(zip(candidate_chunks, similarity_scores))
    ranked_results.sort(key=lambda x: x[1], reverse=True)
    return ranked_results


def search_knowledge_base(query_text: str, top_k: int = 3) -> None:
    print(f"\n================ SEARCH QUERY ================")
    print(f"Query: '{query_text}'\n")

    chroma_results = retrieve_top_k_chunks(query_text, top_k=top_k)

    if not chroma_results:
        print("No matching records found in the vector database.")
        return

    print(f"Top {len(chroma_results)} Matches Retrieved from ChromaDB:")
    for idx, item in enumerate(chroma_results, 1):
        print(f"\n--- Result #{idx} ---")
        print(f"Source: {item['metadata'].get('source', 'Unknown')}")
        print(f"Distance Score: {item['chroma_distance']:.4f}")
        print(f"Content Sample: {item['chunk_text'][:150]}...")

    extracted_texts = [item["chunk_text"] for item in chroma_results]
    ranked_text_scores = rank_chunks_with_cosine_similarity(query_text, extracted_texts)

    print("\n================ COSINE RE-RANKING ================")
    for idx, (text, score) in enumerate(ranked_text_scores, 1):
        print(f"Rank {idx} | Similarity Score: {score:.4f} | Snippet: {text[:80]}...")


if __name__ == "__main__":
    data_directory = "./sample_data"

    # Ensure the directory exists
    os.makedirs(data_directory, exist_ok=True)

    # Ensure sample file is written regardless of existing folder structure
    sample_file_path = os.path.join(data_directory, "sample.txt")
    with open(sample_file_path, "w", encoding="utf-8") as f:
        f.write("Retrieval-Augmented Generation (RAG) combines search algorithms with LLMs.\n" * 20)

    print("--- Starting Data Ingestion Pipeline ---")

    # 1. Load
    raw_docs = load_documents_from_directory(data_directory)

    # 2. Chunk
    doc_chunks = split_documents(raw_docs, chunk_size=300, chunk_overlap=30)

    # 3. Embed & Store
    ingest_chunks_to_chroma(doc_chunks)

    # Verify collection count
    print(f"Total stored vectors in collection: {collection.count()}")

    # 4. Search Execution
    sample_query = "How does Retrieval-Augmented Generation work?"
    search_knowledge_base(query_text=sample_query, top_k=3)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- Starting Data Ingestion Pipeline ---
Loaded 1 raw document page(s) from './sample_data'.
Split 1 documents into 5 text chunks.
Ingested batch 1/1 (5 chunks)

Successfully ingested all 5 chunks into ChromaDB!
Total stored vectors in collection: 5

================ SEARCH QUERY ================
Query: 'How does Retrieval-Augmented Generation work?'

Top 3 Matches Retrieved from ChromaDB:

--- Result #1 ---
Source: sample_data/sample.txt
Distance Score: 0.3549
Content Sample: Retrieval-Augmented Generation (RAG) combines search algorithms with LLMs.
Retrieval-Augmented Generation (RAG) combines search algorithms with LLMs.
...

--- Result #2 ---
Source: sample_data/sample.txt
Distance Score: 0.3549
Content Sample: Retrieval-Augmented Generation (RAG) combines search algorithms with LLMs.
Retrieval-Augmented Generation (RAG) combines search algorithms with LLMs.
...

--- Result #3 ---
Source: sample_data/sample.txt
Distance Score: 0.3549
Content Sample: Retrieval-Augmented Generation (